# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarshadad-codeee/FlyRank_ML_Task1"
REPO_DIR = "FlyRank_ML_Task1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())
!pip install duckdb --quiet

Working directory: /content/FlyRank_ML_Task1


In [2]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

base = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{base}/fact_content_daily_performance/month=2026-03/*.parquet"

In [4]:
content_meta = con.sql(f"""
    SELECT content_hash_id, client_hash_id, content_created_date,
           last_optimized_date, is_published, is_deleted
    FROM read_parquet('{base}/dim_content.parquet')
""").df()

df = features_df.merge(content_meta, on=["content_hash_id", "client_hash_id"], how="left")

df["ctr"] = df["total_clicks"] / df["total_impressions"].replace(0, pd.NA)
df["content_created_date"] = pd.to_datetime(df["content_created_date"])
reference_date = pd.Timestamp("2026-03-31")
df["content_age_days"] = (reference_date - df["content_created_date"]).dt.days

print(f"Merged frame shape: {df.shape}")
print(f"Any negative ages? {(df['content_age_days'] < 0).sum()} rows")
df.head(5)

Merged frame shape: (176738, 11)
Any negative ages? 0 rows


,content_hash_id,client_hash_id,total_clicks,total_impressions,avg_position_proxy,content_created_date,last_optimized_date,is_published,is_deleted,ctr,content_age_days
0,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,1.0,899.0,168.935484,2026-02-12,2026-06-15,True,False,0.001112,47
1,content_4a1ca0fa5c177e0c,client_62f4a7e64f5e0096,0.0,14.0,6.000000,2026-02-12,NaT,True,False,0.000000,47
2,content_c03ecafd4c999f15,client_62f4a7e64f5e0096,22.0,10849.0,2817.193548,2026-02-12,2026-06-15,True,False,0.002028,47
3,content_e689bc511192751a,client_62f4a7e64f5e0096,0.0,61.0,13.296296,2026-02-12,NaT,True,False,0.000000,47
4,content_babcf791dccc1610,client_62f4a7e64f5e0096,0.0,181.0,70.833333,2026-02-12,NaT,True,False,0.000000,47


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.